# Estruturacao -- ydata-profiling antes/depois

In [1]:
import sys
from pathlib import Path

import pandas as pd


def encontrar_raiz_repo(marcadores=(".git", "src")):
    caminho = Path.cwd()
    for pasta in [caminho, *caminho.parents]:
        if any((pasta / m).exists() for m in marcadores):
            return pasta
    raise FileNotFoundError(f"Nao achei nenhum de {marcadores} subindo a partir de {caminho}")


RAIZ = encontrar_raiz_repo()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

pd.set_option("display.max_colwidth", 120)


In [3]:
from ydata_profiling import ProfileReport

/home/rodrigo.lusa/.conda/envs/ydata_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Carregando `raw_features.csv` (saida de `03_new_make_features.ipynb`)

In [22]:
df = pd.read_csv("../data/processed/features.csv", index_col=0)
print(f"{len(df)} genomas, {df.shape[1]} features")
df.head()


2230 genomas, 10 features


,n_beta_lactam_total,pct_esbl,pct_A_serino_carbapenemase,pct_BD_carbapenemase,pct_C_ampc,pct_nao_hidrolise,pct_beta_lactam_plasmidial,n_aminoglicosideo,n_polimixina,gc_diff_plasmid_cromossomo
sample,,,,,,,,,,
GCA_000316425.1,15,0.000000,0.0,0.0,0.066667,0.933333,0.133333,0,4,-2.92
GCA_000355215.1,16,0.000000,0.0,0.0,0.062500,0.937500,0.125000,2,3,-2.10
GCA_000355195.1,16,0.000000,0.0,0.0,0.062500,0.937500,0.125000,2,3,-1.50
GCA_000355235.1,17,0.058824,0.0,0.0,0.058824,0.882353,0.058824,2,4,-1.13
GCA_000355255.1,17,0.058824,0.0,0.0,0.058824,0.882353,0.176471,0,4,-1.17


## Profile antes

In [5]:
profile_antes = ProfileReport(df, title="Features -- pre-estruturacao", minimal=False)
profile_antes.to_file("../reports/features_antes_estruturacao.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 50.99it/s]


## Estruturacao

1. **NaN** -- `pct_*` vira NaN so quando `n_beta_lactam_total == 0` (genoma sem hit beta-lactamico pra
   normalizar). `fillna(0)` -- leitura: "sem base de calculo, tratado como sem essa resistencia
   especifica".
2. **Duplicatas** -- `drop_duplicates()`.
3. **Desbalanceamento** -- colunas com >80% zero viram tambem uma versao binaria (presenca/ausencia),
   ADICIONADA (nao substitui a proporcao continua) -- mesmo tipo de transformacao que o notebook
   original fez em `prop_carbapenemase_BD_plasmidial`.

In [23]:
print(f"NaN antes: {df.isna().sum().sum()}")
df_estruturado = df.fillna(0)

NaN antes: 0


In [24]:
corr = df_estruturado.select_dtypes(include='number').corr()
limite = 0.5
corr[corr > limite].style.background_gradient(cmap='viridis')


,n_beta_lactam_total,pct_esbl,pct_A_serino_carbapenemase,pct_BD_carbapenemase,pct_C_ampc,pct_nao_hidrolise,pct_beta_lactam_plasmidial,n_aminoglicosideo,n_polimixina,gc_diff_plasmid_cromossomo
n_beta_lactam_total,1.000000,nan,nan,nan,nan,nan,nan,nan,0.783139,nan
pct_esbl,nan,1.000000,nan,nan,nan,nan,0.500940,nan,nan,nan
pct_A_serino_carbapenemase,nan,nan,1.000000,nan,nan,nan,nan,nan,nan,nan
pct_BD_carbapenemase,nan,nan,nan,1.000000,0.673074,nan,nan,nan,nan,nan
pct_C_ampc,nan,nan,nan,0.673074,1.000000,nan,nan,nan,nan,nan
pct_nao_hidrolise,nan,nan,nan,nan,nan,1.000000,nan,nan,0.747154,nan
pct_beta_lactam_plasmidial,nan,0.500940,nan,nan,nan,nan,1.000000,nan,nan,nan
n_aminoglicosideo,nan,nan,nan,nan,nan,nan,nan,1.000000,nan,nan
n_polimixina,0.783139,nan,nan,nan,nan,0.747154,nan,nan,1.000000,nan
gc_diff_plasmid_cromossomo,nan,nan,nan,nan,nan,nan,nan,nan,nan,1.000000


#### Correlações
- n_beta_lactam_total <-> n_polimixina
- pct_nao_hidrolise <-> n_polimixina
- pct_BD_carbapenemase <-> pct_C_ampc

Remover n_polimixina, por mais que seja biologicamente relevante (>0.75)

In [18]:
df_estruturado.columns

Index(['n_beta_lactam_total', 'pct_esbl', 'pct_A_serino_carbapenemase',
       'pct_BD_carbapenemase', 'pct_C_ampc', 'pct_nao_hidrolise',
       'pct_beta_lactam_plasmidial', 'n_aminoglicosideo', 'n_polimixina',
       'gc_diff_plasmid_cromossomo'],
      dtype='object')

In [25]:
df_estruturado = df_estruturado[['n_beta_lactam_total', 'pct_esbl', 'pct_A_serino_carbapenemase',
       'pct_BD_carbapenemase', 'pct_C_ampc', 'pct_nao_hidrolise',
       'pct_beta_lactam_plasmidial', 'n_aminoglicosideo',
       'gc_diff_plasmid_cromossomo']]

In [26]:
n_antes = len(df_estruturado)
df_estruturado = df_estruturado.drop_duplicates(keep='first')
print(f"Duplicatas removidas: {n_antes - len(df_estruturado)}")

Duplicatas removidas: 82


## Profile depois

In [27]:
profile_depois = ProfileReport(df_estruturado, title="Features -- pos-estruturacao", minimal=False)
profile_depois.to_file("../reports/features_depois_estruturacao.html")

Summarize dataset:  50%|█████     | 7/14 [00:00<00:00, 66.56it/s, Describe variable: gc_diff_plasmid_cromossomo]


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 75.37it/s]


## Salvando pra proxima etapa (`05_divisao.ipynb`)

In [28]:
df_estruturado.to_csv("../data/processed/features_estruturadas.csv")
print(f"Salvo: {'../data/processed/features_estruturadas.csv'} -- {df_estruturado.shape}")


Salvo: ../data/processed/features_estruturadas.csv -- (2148, 9)
